# 01 -- Stage 1: Train on Image Datasets

Trains `ForensicMultiHeadNet` on `data/labels.csv` (CASIA2.0, CoMoFoD,
Coverage, IMD2020, FaceForensics++ images -- combine them into one
`labels.csv` + `images/`/`masks/` before running this). Saves
`models/image_model.pth`.

In [ ]:
import sys, os
PROJECT_ROOT = "/content/ForensicAI_Lab"  # adjust if you cloned/uploaded elsewhere
if not os.path.exists(PROJECT_ROOT):
    PROJECT_ROOT = os.getcwd()  # fallback: running locally from the project root
sys.path.insert(0, PROJECT_ROOT)

from notebooks.utils.colab_common import mount_drive, install_requirements, is_colab

if is_colab():
    mount_drive()
    install_requirements(PROJECT_ROOT)
else:
    print("Not in Colab -- assuming requirements-train.txt is already installed locally.")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import cv2

from config.loader import get_settings, resolve_dataset_root, resolve_path
from services.preprocessing.pipeline import preprocess_image
from services.detection.model import ForensicMultiHeadNet

settings = get_settings()
dataset_root = resolve_dataset_root(settings)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", device)

In [ ]:
class ForensicImageDataset(Dataset):
    def __init__(self, labels_csv, images_dir, masks_dir, image_size):
        self.df = pd.read_csv(labels_csv)
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.image_size = image_size
        self.label_to_idx = {l: i for i, l in enumerate(settings.labels.binary)}
        manip_classes = settings.labels.manipulation_types
        self.manip_to_idx = {m: i for i, m in enumerate(manip_classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_bgr = cv2.imread(str(self.images_dir / row["filename"]))
        x = preprocess_image(image_bgr, tuple(self.image_size))

        label_idx = self.label_to_idx[row["label"]]
        manip_idx = self.manip_to_idx.get(row["type"], 0)

        mask_path = self.masks_dir / row["filename"]
        if mask_path.exists():
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(mask, tuple(self.image_size))
            mask = (mask > 127).astype(np.float32)[np.newaxis, :, :]
            has_mask = 1.0
        else:
            mask = np.zeros((1, *self.image_size), dtype=np.float32)
            has_mask = 0.0

        return (
            torch.from_numpy(x),
            torch.tensor(label_idx, dtype=torch.long),
            torch.tensor(manip_idx, dtype=torch.long),
            torch.from_numpy(mask),
            torch.tensor(has_mask, dtype=torch.float32),
        )

In [ ]:
labels_csv = dataset_root / settings.data.labels_csv
images_dir = dataset_root / settings.data.images_dir
masks_dir = dataset_root / settings.data.masks_dir

dataset = ForensicImageDataset(labels_csv, images_dir, masks_dir, settings.inference.image_size)
loader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=2)
print("Dataset size:", len(dataset))

In [ ]:
model = ForensicMultiHeadNet(
    backbone=settings.models.backbone,
    in_chans=settings.inference.input_channels,
    num_manip_classes=len(settings.labels.manipulation_types),
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
cls_loss_fn = torch.nn.CrossEntropyLoss()
manip_loss_fn = torch.nn.CrossEntropyLoss()
mask_loss_fn = torch.nn.BCEWithLogitsLoss(reduction="none")

NUM_EPOCHS = 5  # adjust based on dataset size / Colab session limits

In [ ]:
model.train()
for epoch in range(NUM_EPOCHS):
    running_loss = 0.0
    for x, label_idx, manip_idx, mask, has_mask in loader:
        x, label_idx, manip_idx = x.to(device), label_idx.to(device), manip_idx.to(device)
        mask, has_mask = mask.to(device), has_mask.to(device)

        optimizer.zero_grad()
        out = model(x)

        loss_cls = cls_loss_fn(out["cls_logits"], label_idx)
        loss_manip = manip_loss_fn(out["manip_logits"], manip_idx)
        mask_loss_per_px = mask_loss_fn(out["mask_logits"], mask)
        loss_mask = (mask_loss_per_px.mean(dim=[1, 2, 3]) * has_mask).sum() / (has_mask.sum() + 1e-8)

        loss = loss_cls + loss_manip + loss_mask
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} -- loss: {running_loss / len(loader):.4f}")

In [ ]:
checkpoint_path = resolve_path(settings.models.image_model_checkpoint)
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), checkpoint_path)
print("Saved checkpoint to", checkpoint_path)